<a href="https://colab.research.google.com/github/TPhan506/Facial-Expression-Classification/blob/main/Facial_Expression_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Facial Expression Classification with PyTorch



In [11]:
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


In [12]:
# Read the 48x48 grayscale images and the three expression labels.
root = Path("/content")
needed = ("train_data.csv", "train_target.csv", "test_data.csv", "sample-submission.csv")
folders = [root / "facial_expression_data", root, root / "sample_data"]
data_dir = next((p for p in folders if all((p / f).exists() for f in needed)), None)

if data_dir is None:
    existing = list(root.rglob("train_data.csv"))
    data_dir = next((p.parent for p in existing if all((p.parent / f).exists() for f in needed)), None)

if data_dir is None:
    zips = list(root.rglob("facial_expression_data.zip"))
    if not zips:
        raise FileNotFoundError("Dataset ZIP or CSV files not found under /content")
    data_dir = root / "facial_expression_data"
    data_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(data_dir)
    data_dir = next(p.parent for p in data_dir.rglob("train_data.csv") if all((p.parent / f).exists() for f in needed))

X = pd.read_csv(data_dir / "train_data.csv", header=None, dtype=np.uint8).to_numpy()
y = pd.read_csv(data_dir / "train_target.csv", header=None).iloc[:, 0].to_numpy()
X_test = pd.read_csv(data_dir / "test_data.csv", header=None, dtype=np.uint8).to_numpy()
submission = pd.read_csv(data_dir / "sample-submission.csv")
classes = sorted(np.unique(y).tolist())
labels = np.array([classes.index(v) for v in y])
assert X.shape[1] == X_test.shape[1] == 2304
assert len(X) == len(y) and len(X_test) == len(submission)
print("Training images:", len(X), "Test images:", len(X_test), "Classes:", classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Training images: 16175 Test images: 3965 Classes: [0, 1, 2]
Using: cuda


## Preprocessing

In [13]:
class Faces(Dataset):
    def __init__(self, images, targets=None, transform=None):
        self.images, self.targets, self.transform = images, targets, transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        image = Image.fromarray(self.images[i].reshape(48, 48)).convert("RGB")
        image = self.transform(image)
        if self.targets is None:
            return image
        return image, int(self.targets[i])

# ImageNet normalization matches the pretrained EfficientNet weights.
normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(), normalize
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), normalize
])

train_ids, val_ids = train_test_split(
    np.arange(len(X)), test_size=0.15, random_state=42, stratify=labels
)
train_loader = DataLoader(Faces(X[train_ids], labels[train_ids], train_tf),
                          batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(Faces(X[val_ids], labels[val_ids], val_tf),
                        batch_size=32, num_workers=2)
test_loader = DataLoader(Faces(X_test, transform=val_tf),
                         batch_size=32, num_workers=2)

## Model and training

In [ ]:
# Fine-tune EfficientNet-B0 after replacing its classifier with a 3-class output.
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))
model = model.to(device)

EPOCHS = 6
PATIENCE = 5
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0002, weight_decay=0.0001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp)

# Training updates weights; validation measures performance on unseen examples.
def run_epoch(loader, training):
    model.train(training)
    loss_total = correct = n = 0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, enabled=amp):
                logits = model(images)
                loss = criterion(logits, targets)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        n += targets.size(0)
        loss_total += loss.item() * targets.size(0)
        correct += (logits.argmax(1) == targets).sum().item()
    return loss_total / n, correct / n

output_dir = Path("/content/facial_expression_output")
output_dir.mkdir(exist_ok=True)
best_acc, wait, history = 0, 0, []
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, True)
    val_loss, val_acc = run_epoch(val_loader, False)
    scheduler.step()
    history.append([epoch, train_loss, train_acc, val_loss, val_acc])
    print(f"Epoch {epoch}/{EPOCHS} | train {train_acc:.3f} | validation {val_acc:.3f}")
    if val_acc > best_acc:
        best_acc, wait = val_acc, 0
        torch.save(model.state_dict(), output_dir / "best_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            break

model.load_state_dict(torch.load(output_dir / "best_model.pt", map_location=device, weights_only=True))
print(f"Best validation accuracy: {best_acc:.2%}")

Epoch 1/6 | train 0.732 | validation 0.803


## Results and predictions

In [ ]:
# Validation curves and confusion matrix.
results = pd.DataFrame(history, columns=["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])
results.to_csv(output_dir / "training_history.csv", index=False)
plt.plot(results.epoch, results.train_acc, label="Training")
plt.plot(results.epoch, results.val_acc, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / "accuracy_plot.png", dpi=150)
plt.show()

model.eval()
actual, predicted = [], []
with torch.no_grad():
    for images, targets in val_loader:
        pred = model(images.to(device)).argmax(1).cpu().numpy()
        actual.extend(targets.numpy())
        predicted.extend(pred)
ConfusionMatrixDisplay(confusion_matrix(actual, predicted), display_labels=classes).plot(cmap="Blues")
plt.tight_layout()
plt.savefig(output_dir / "confusion_matrix.png", dpi=150)
plt.show()

# Test-time horizontal flipping averages predictions from two image orientations.
test_preds = []
with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        with torch.autocast(device_type=device.type, enabled=amp):
            p1 = model(images).softmax(1)
            p2 = model(torch.flip(images, dims=[3])).softmax(1)
        test_preds.extend(((p1 + p2) / 2).argmax(1).cpu().tolist())

submission["Category"] = [classes[i] for i in test_preds]
submission.to_csv(output_dir / "submission.csv", index=False)
print("Best validation accuracy:", f"{best_acc:.2%}")
print("Saved:", output_dir / "submission.csv")
print(submission.head())